# 20 — Generate `usdm_v4.ttl`

Mechanical conversion of `../downloads/dataStructure.yml` to `../usdm_v4.ttl`.

**Mapping summary** (full table in `README.md`):

| Source (YAML)                   | Target (RDF/OWL)                                                  |
|---------------------------------|-------------------------------------------------------------------|
| Top-level entry                 | `owl:Class` with `rdfs:label`                                     |
| `NCI C-Code`                    | `skos:exactMatch ncit:Cxxxxx` + `skos:prefLabel` + `skos:definition` |
| `Super Classes`                 | `rdfs:subClassOf`                                                 |
| `Modifier`                      | `usdm:modifier`                                                   |
| Attribute (primitive `$ref`)    | `owl:DatatypeProperty`, name `{Class}-{attr}`                     |
| Attribute (class `$ref`)        | `owl:ObjectProperty`, name `{Class}-{attr}`                       |
| Multi-`$ref` Type               | `rdfs:range` with `owl:unionOf`                                   |
| `Cardinality`                   | `owl:Restriction` (`owl:cardinality`/`owl:minCardinality`/`owl:maxCardinality`); omit `0..*`/`*` |
| `Relationship Type` / `Model Name` / `Model Representation` | `usdm:relationshipType` / `usdm:modelName` / `usdm:modelRepresentation` |

Inherited attributes (`Inherited From` present) are **not** re-emitted on the inheriting class.


## 1. Load source YAML


In [1]:
import yaml
from pathlib import Path

YAML_PATH = "../downloads/dataStructure.yml"

if not Path(YAML_PATH).exists():
    raise FileNotFoundError(
        f"{YAML_PATH} missing — run notebooks/10_fetch_yaml.ipynb first."
    )

with open(YAML_PATH) as f:
    SOURCE = yaml.safe_load(f)

print(f"loaded {len(SOURCE)} top-level entries from {YAML_PATH}")


loaded 86 top-level entries from ../downloads/dataStructure.yml


## 1b. Load USDM_CT bindings

For each row in sheet `DDF Entities&Attributes` where `Has Value List`
starts with `Y`, derive `(entity, attr_name, raw_hvl, codelist_ccode)`.
Codelist C-code is read from the `Codelist URL` column
(`.../subset/ncit/Cxxxxxx`) when populated; cross-validated against any
`Cxxxxxx` extractable from the `Has Value List` cell text. Disagreement
raises.

Fail-fast: every Y-row must resolve to a YAML attribute whose `Type` list
references `Code`, `AliasCode`, or `ResponseCode`.
Rows where `Inherited From` is non-null are skipped — the property IRI
exists only on the declaring class per v0's mechanical mapping, and the
inherited row duplicates the parent's binding.


In [2]:
import openpyxl
import re
from pathlib import Path

CT_PATH = "../downloads/USDM_CT.xlsx"
if not Path(CT_PATH).exists():
    raise FileNotFoundError(
        f"{CT_PATH} missing — run notebooks/10_fetch_yaml.ipynb first."
    )

CODE_TYPED_TARGETS = {"Code", "AliasCode", "ResponseCode"}
RE_URL_CCODE  = re.compile(r"/subset/ncit/(C\d+)\s*$")
RE_TEXT_CCODE = re.compile(r"\b(C\d+)\b")

wb = openpyxl.load_workbook(CT_PATH, read_only=True, data_only=True)
ws = wb["DDF Entities&Attributes"]
rows = list(ws.iter_rows(values_only=True))
header = list(rows[0])
i_entity = header.index("Entity Name")
i_attr   = header.index("Logical Data Model Name")
i_hvl    = header.index("Has Value List")
i_url    = header.index("Codelist URL")
i_inh    = header.index("Inherited From")

CT_BINDINGS = {}
for r in rows[1:]:
    if r[i_inh]:
        continue
    hvl = r[i_hvl]
    if not hvl or not str(hvl).startswith("Y"):
        continue
    ent = r[i_entity]
    att = r[i_attr]
    url = r[i_url]
    ccode_url, ccode_text = None, None
    if url:
        m = RE_URL_CCODE.search(str(url))
        if m: ccode_url = m.group(1)
    text_codes = RE_TEXT_CCODE.findall(str(hvl))
    if len(text_codes) > 1:
        raise ValueError(f"multiple C-codes in HVL for {ent}.{att}: {hvl!r}")
    if text_codes:
        ccode_text = text_codes[0]
    if ccode_url and ccode_text and ccode_url != ccode_text:
        raise ValueError(
            f"C-code mismatch for {ent}.{att}: url={ccode_url} text={ccode_text} hvl={hvl!r}"
        )
    ccode = ccode_url or ccode_text
    CT_BINDINGS[(ent, att)] = {
        "raw_hvl": str(hvl),
        "ccode":   ccode,
    }

n_total      = len(CT_BINDINGS)
n_structured = sum(1 for v in CT_BINDINGS.values() if v["ccode"])
n_freetext   = n_total - n_structured
print(f"CT_BINDINGS: {n_total} Y-rows; {n_structured} structured (Cxxxxxx); {n_freetext} free-text")


CT_BINDINGS: 57 Y-rows; 45 structured (Cxxxxxx); 12 free-text


In [3]:
mismatches = []
type_failures = []
for (ent, att), b in CT_BINDINGS.items():
    if ent not in SOURCE:
        mismatches.append((ent, att, "entity not in YAML"))
        continue
    attrs = SOURCE[ent].get("Attributes") or {}
    if att not in attrs:
        mismatches.append((ent, att, "attribute not on entity"))
        continue
    a = attrs[att]
    targets = [t.get("$ref","").lstrip("#/") for t in (a.get("Type") or [])]
    if not (len(targets) == 1 and targets[0] in CODE_TYPED_TARGETS):
        type_failures.append((ent, att, targets))

if mismatches:
    raise RuntimeError(f"CT_BINDINGS join failures: {mismatches}")
if type_failures:
    raise RuntimeError(f"CT_BINDINGS type failures (not Code/AliasCode/ResponseCode): {type_failures}")

print(f"all {len(CT_BINDINGS)} CT bindings resolve to a Code-typed YAML attribute")


all 57 CT bindings resolve to a Code-typed YAML attribute


## 2. Constants and prefix table


In [4]:
ONTOLOGY_IRI = "https://w3id.org/cdisc/usdm/v4"
USDM_NS      = ONTOLOGY_IRI + "#"
NCIT_NS      = "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#"
TARGET_PATH  = "../usdm_v4.ttl"
META_PATH    = "../downloads/.fetch_meta.json"

PRIMITIVE_TO_XSD = {
    "string":  "xsd:string",
    "boolean": "xsd:boolean",
    "date":    "xsd:date",
    "float":   "xsd:float",
    "integer": "xsd:integer",
}

PREFIXES = [
    ("owl",     "http://www.w3.org/2002/07/owl#"),
    ("rdf",     "http://www.w3.org/1999/02/22-rdf-syntax-ns#"),
    ("rdfs",    "http://www.w3.org/2000/01/rdf-schema#"),
    ("skos",    "http://www.w3.org/2004/02/skos/core#"),
    ("xsd",     "http://www.w3.org/2001/XMLSchema#"),
    ("dcterms", "http://purl.org/dc/terms/"),
    ("ncit",    NCIT_NS),
    ("usdm",    USDM_NS),
]

ANNOTATION_PROPERTIES = ["modifier", "relationshipType", "modelName", "modelRepresentation", "boundCodelist", "boundCodelistNote"]


## 3. Turtle string helpers

Pure functions — no I/O. Produce well-formed Turtle fragments.


In [5]:
import re

_SAFE_LOCAL = re.compile(r"^[A-Za-z][A-Za-z0-9_-]*$")

def safe_local(name):
    if not _SAFE_LOCAL.match(name):
        raise ValueError(f"unsafe local name: {name!r}")
    return name

def quote_literal(s):
    """Escape a Python string for use as a Turtle string literal."""
    s = s.replace("\\", "\\\\").replace('"', '\\"')
    s = s.replace("\n", "\\n").replace("\r", "\\r").replace("\t", "\\t")
    return f'"{s}"'

def class_qname(name):
    return f"usdm:{safe_local(name)}"

def property_qname(class_name, attr_name):
    return f"usdm:{safe_local(class_name)}-{safe_local(attr_name)}"

def nci_qname(c_code):
    return f"ncit:{safe_local(c_code)}"

def resolve_ref(ref):
    """Resolve a JSON-pointer-style $ref like '#/Foo' to the target name."""
    if not ref.startswith("#/"):
        raise ValueError(f"unexpected $ref form: {ref!r}")
    return ref[2:]

def parse_cardinality(card):
    """Return list of (predicate, integer_value) for cardinality, or [] if open."""
    if card == "1":
        return [("owl:cardinality", 1)]
    if card in ("0..*", "*"):
        return []
    if ".." not in card:
        raise ValueError(f"unexpected cardinality: {card!r}")
    lo, hi = card.split("..", 1)
    out = []
    if lo == "1":
        out.append(("owl:minCardinality", 1))
    elif lo != "0":
        raise ValueError(f"unexpected cardinality lower bound: {card!r}")
    if hi != "*":
        out.append(("owl:maxCardinality", int(hi)))
    return out

def render_block(subject, triples):
    """Render Turtle predicate-object list. triples = [(pred_qname, obj_str), ...]."""
    if not triples:
        raise ValueError(f"empty triple list for subject {subject}")
    lines = [subject]
    n = len(triples)
    for i, (pred, obj) in enumerate(triples):
        terminator = " ." if i == n - 1 else " ;"
        lines.append(f"    {pred} {obj}{terminator}")
    return lines


## 4. Emit ontology header

Reads `../downloads/.fetch_meta.json` written by notebook 10 to record the
DDF-RA tag and source URL on the ontology resource.


In [6]:
import json

with open(META_PATH) as f:
    META = json.load(f)

header_lines = []
for pfx, ns in PREFIXES:
    header_lines.append(f"@prefix {pfx}: <{ns}> .")
header_lines.append(f"@base <{ONTOLOGY_IRI}> .")
header_lines.append("")

ontology_triples = [
    ("a", "owl:Ontology"),
    ("dcterms:source", f"<{META['raw_url']}>"),
    ("owl:versionInfo", quote_literal(META["ddf_ra_tag"])),
    ("rdfs:comment", quote_literal(
        "Mechanical conversion of CDISC USDM v4 dataStructure.yml. Draft — not normative."
    )),
]
header_lines += render_block(f"<{ONTOLOGY_IRI}>", ontology_triples)
header_lines.append("")

for ann in ANNOTATION_PROPERTIES:
    header_lines.append(f"usdm:{ann} a owl:AnnotationProperty .")
header_lines.append("")

print("\n".join(header_lines))


@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ncit: <http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#> .
@prefix usdm: <https://w3id.org/cdisc/usdm/v4#> .
@base <https://w3id.org/cdisc/usdm/v4> .

<https://w3id.org/cdisc/usdm/v4>
    a owl:Ontology ;
    dcterms:source <https://raw.githubusercontent.com/cdisc-org/DDF-RA/v4.0.0/Deliverables/UML/dataStructure.yml> ;
    owl:versionInfo "v4.0.0" ;
    rdfs:comment "Mechanical conversion of CDISC USDM v4 dataStructure.yml. Draft — not normative." .

usdm:modifier a owl:AnnotationProperty .
usdm:relationshipType a owl:AnnotationProperty .
usdm:modelName a owl:AnnotationProperty .
usdm:modelRepresentation a owl:AnnotationProperty .
usdm:boundCodelist a owl:Annotat

## 5. Emit classes and properties

For each top-level YAML entry, emit the class block (with cardinality
restrictions on non-inherited attributes) and one property block per
non-inherited attribute. Inherited attributes are skipped — they belong
to the parent class.

`skos:exactMatch` and `skos:prefLabel` are emitted only when an `NCI C-Code`
is present (mirrors CDISC Library RDF convention). `skos:definition` is
emitted whenever a `Definition` field is present in source, whether or not
the entity is NCIt-anchored — "preserve, don't collapse".


In [7]:
def class_triples(name, entry):
    triples = [
        ("a", "owl:Class"),
        ("rdfs:label", quote_literal(name)),
    ]
    if "NCI C-Code" in entry:
        triples.append(("skos:exactMatch", nci_qname(entry["NCI C-Code"])))
        if "Preferred Term" in entry:
            triples.append(("skos:prefLabel", quote_literal(entry["Preferred Term"])))
    if "Definition" in entry:
        triples.append(("skos:definition", quote_literal(entry["Definition"])))
    for s in (entry.get("Super Classes") or []):
        triples.append(("rdfs:subClassOf", class_qname(resolve_ref(s["$ref"]))))
    if "Modifier" in entry:
        triples.append(("usdm:modifier", quote_literal(entry["Modifier"])))
    for aname, attr in (entry.get("Attributes") or {}).items():
        if not isinstance(attr, dict) or "Inherited From" in attr:
            continue
        prop_q = property_qname(name, aname)
        for pred, val in parse_cardinality(attr.get("Cardinality")):
            restriction = (
                f"[ a owl:Restriction ; owl:onProperty {prop_q} ; "
                f"{pred} \"{val}\"^^xsd:nonNegativeInteger ]"
            )
            triples.append(("rdfs:subClassOf", restriction))
    return triples

def property_triples(class_name, attr_name, attr):
    targets = [resolve_ref(r["$ref"]) for r in attr["Type"]]
    is_primitive = all(t in PRIMITIVE_TO_XSD for t in targets)
    is_class = all(t not in PRIMITIVE_TO_XSD for t in targets)
    if not (is_primitive or is_class):
        raise ValueError(f"mixed primitive+class targets on {class_name}.{attr_name}: {targets}")
    triples = [
        ("a", "owl:DatatypeProperty" if is_primitive else "owl:ObjectProperty"),
        ("rdfs:label", quote_literal(f"{class_name}-{attr_name}")),
        ("rdfs:domain", class_qname(class_name)),
    ]
    if len(targets) == 1:
        t = targets[0]
        rng = PRIMITIVE_TO_XSD[t] if t in PRIMITIVE_TO_XSD else class_qname(t)
        triples.append(("rdfs:range", rng))
    else:
        union_members = " ".join(class_qname(t) for t in targets)
        triples.append(("rdfs:range", f"[ a owl:Class ; owl:unionOf ( {union_members} ) ]"))
    if "NCI C-Code" in attr:
        triples.append(("skos:exactMatch", nci_qname(attr["NCI C-Code"])))
        if "Preferred Term" in attr:
            triples.append(("skos:prefLabel", quote_literal(attr["Preferred Term"])))
    if "Definition" in attr:
        triples.append(("skos:definition", quote_literal(attr["Definition"])))
    if "Relationship Type" in attr:
        triples.append(("usdm:relationshipType", quote_literal(attr["Relationship Type"])))
    if "Model Name" in attr:
        triples.append(("usdm:modelName", quote_literal(attr["Model Name"])))
    if "Model Representation" in attr:
        triples.append(("usdm:modelRepresentation", quote_literal(attr["Model Representation"])))
    binding = CT_BINDINGS.get((class_name, attr_name))
    if binding is not None:
        if binding["ccode"]:
            triples.append(("usdm:boundCodelist", nci_qname(binding["ccode"])))
        triples.append(("usdm:boundCodelistNote", quote_literal(binding["raw_hvl"])))
    return triples

body_lines = []
n_classes = 0
n_props = 0
for cname, entry in SOURCE.items():
    body_lines.append(f"# class {cname}")
    body_lines += render_block(class_qname(cname), class_triples(cname, entry))
    body_lines.append("")
    n_classes += 1
    for aname, attr in (entry.get("Attributes") or {}).items():
        if not isinstance(attr, dict) or "Inherited From" in attr:
            continue
        body_lines += render_block(property_qname(cname, aname), property_triples(cname, aname, attr))
        body_lines.append("")
        n_props += 1

print(f"emitted {n_classes} classes and {n_props} properties")


emitted 86 classes and 693 properties


## 6. Write `usdm_v4.ttl` atomically

Write to `usdm_v4.ttl.tmp`, then rename. A failed run never leaves a
half-finished deliverable.


In [8]:
import os
from pathlib import Path

all_lines = header_lines + body_lines
ttl = "\n".join(all_lines).rstrip("\n") + "\n"

tmp = Path(TARGET_PATH + ".tmp")
tmp.write_text(ttl, encoding="utf-8")
os.replace(tmp, TARGET_PATH)

print(f"wrote {len(ttl)} bytes ({len(all_lines)} lines) to {TARGET_PATH}")


wrote 373956 bytes (8324 lines) to ../usdm_v4.ttl


## 7. Smoke test — parse with rdflib

Quick sanity check that the file is valid Turtle and triples parse.
Full validation lives in `30_validate.ipynb`.


In [9]:
import rdflib

g = rdflib.Graph()
g.parse(TARGET_PATH, format="turtle")
print(f"parsed {len(g)} triples from {TARGET_PATH}")


parsed 8173 triples from ../usdm_v4.ttl


## Provenance

Source: `cdisc-org/DDF-RA@<tag>` at `Deliverables/UML/dataStructure.yml`.
Tag is recorded in `../downloads/.fetch_meta.json` and on `<ontology>` via
`owl:versionInfo` and `dcterms:source`.
